In [17]:
import pandas as pd 
import numpy as np 
import matplotlib 
from pathlib import Path

# Model Selection and Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# for hyperparam tuning
from sklearn.model_selection import GridSearchCV

In [18]:
data_path = Path('../data/preprocessed_pokemon_data.csv')
pkmn_df = pd.read_csv(data_path)
seed = 42

In [19]:
target = ['tier']
X = pkmn_df.drop(columns=[t for t in target])
y = pkmn_df[target]

# 80% train 20% test, stratify is to make sure the tier distribution is more even in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

In [15]:
'''
NOTE: objective: multi-softclass essentially tells the model it is doing a classification problem
given multiple classes like OU, UU, etc. choose the most fitting one and output; softmax converts tier scores->probabilities

'''

tier_model = XGBClassifier(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth=6,
    colsample_bytree=0.3,
    num_class=len(np.unique(y))
)

tier_model.fit(X_train, y_train)
predictions = tier_model.predict(X_test)


print(classification_report(y_test, predictions, zero_division=0))

mae = mean_absolute_error(y_test, predictions)
print(mae)


              precision    recall  f1-score   support

           0       0.86      0.78      0.82        46
           1       0.90      1.00      0.95        64
           2       0.85      0.57      0.68        30
           3       0.74      0.90      0.81        93
           4       0.25      0.09      0.13        11
           5       0.17      0.14      0.15        14
           6       0.71      0.42      0.53        12

    accuracy                           0.77       270
   macro avg       0.64      0.56      0.58       270
weighted avg       0.76      0.77      0.76       270

0.45185185185185184


In [16]:
tier_rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    max_features='sqrt',
    random_state=seed,
    class_weight='balanced_subsample' # chose because some tiers have very few mons
)

tier_rf.fit(X_train, y_train.values.ravel())
rf_predictions = tier_rf.predict(X_test)

# 4. EVALUATE
print("--- Random Forest Tier Prediction Results ---")
print(classification_report(y_test, rf_predictions, zero_division=0))

print(mean_absolute_error(y_test, rf_predictions))

--- Random Forest Tier Prediction Results ---
              precision    recall  f1-score   support

           0       0.76      0.57      0.65        46
           1       0.85      1.00      0.92        64
           2       0.81      0.57      0.67        30
           3       0.68      0.90      0.77        93
           4       0.00      0.00      0.00        11
           5       0.29      0.14      0.19        14
           6       0.75      0.50      0.60        12

    accuracy                           0.74       270
   macro avg       0.59      0.53      0.54       270
weighted avg       0.70      0.74      0.71       270

0.5814814814814815


In [10]:
print(y_test.value_counts())
tier_order = ['Illegal','LC', 'NFE', 'RU', 'UU', 'OU', 'Uber', 'AG']
print(list(zip(tier_order, [i for i in range(10)])))

tier
3       93
1       64
0       46
2       30
5       14
6       12
4       11
Name: count, dtype: int64
[('Illegal', 0), ('LC', 1), ('NFE', 2), ('RU', 3), ('UU', 4), ('OU', 5), ('Uber', 6), ('AG', 7)]
